<a href="https://colab.research.google.com/github/cahyoadi88/Script/blob/main/Copy_of_P9_indonesian_news_title.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import gdown
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense,Embedding,LSTM,Dropout

In [ ]:
# URL file yang akan diunduh
file_url = 'https://drive.google.com/u/0/uc?id=1cTdHP5Z8FTQUVRt8QGiGrePUmkvCinmk'

# Path tempat Anda ingin menyimpan file yang diunduh
output_path = '/content/sample_data/indonesian-news-title.csv'

gdown.download(file_url, output_path, quiet=False)


Downloading...
From: https://drive.google.com/u/0/uc?id=1cTdHP5Z8FTQUVRt8QGiGrePUmkvCinmk
To: /content/sample_data/indonesian-news-title.csv
100%|██████████| 16.1M/16.1M [00:00<00:00, 23.0MB/s]


'/content/sample_data/indonesian-news-title.csv'

In [ ]:
# mendefinisikan dataset yang akan digunakan serta menampilkan informasi tipe data dalam dataset
df = pd.read_csv('/content/sample_data/indonesian-news-title.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91017 entries, 0 to 91016
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   date      91017 non-null  object
 1   url       91017 non-null  object
 2   title     91017 non-null  object
 3   category  91017 non-null  object
dtypes: object(4)
memory usage: 2.8+ MB


In [ ]:
df.head()

,date,url,title,category
0,02/26/2020,https://finance.detik.com/berita-ekonomi-bisni...,Kemnaker Awasi TKA di Meikarta,finance
1,02/26/2020,https://finance.detik.com/berita-ekonomi-bisni...,BNI Digitalkan BNI Java Jazz 2020,finance
2,02/26/2020,https://finance.detik.com/berita-ekonomi-bisni...,"Terbang ke Australia, Edhy Prabowo Mau Genjot ...",finance
3,02/26/2020,https://finance.detik.com/moneter/d-4916133/oj...,OJK Siapkan Stimulus Ekonomi Antisipasi Dampak...,finance
4,02/26/2020,https://finance.detik.com/berita-ekonomi-bisni...,Saran Buat Anies-RK yang Mangkir Rapat Banjir ...,finance


In [ ]:
# menghapus kolom yang tidak diperlukan
df = df.drop(columns='date')
df = df.drop(columns='url')
df

,title,category
0,Kemnaker Awasi TKA di Meikarta,finance
1,BNI Digitalkan BNI Java Jazz 2020,finance
2,"Terbang ke Australia, Edhy Prabowo Mau Genjot ...",finance
3,OJK Siapkan Stimulus Ekonomi Antisipasi Dampak...,finance
4,Saran Buat Anies-RK yang Mangkir Rapat Banjir ...,finance
...,...,...
91012,"Ketumpahan Air Panas di Pesawat, Kamu Bisa Tun...",travel
91013,Foto: Bali & 9 Destinasi Paling Instagramable ...,travel
91014,Game Bikin Turis Ini Liburan ke Jepang untuk.....,travel
91015,"Sekeluarga Didepak dari Pesawat, Maskapai Bila...",travel


In [ ]:
print(df['title'][4])

Saran Buat Anies-RK yang Mangkir Rapat Banjir di DPR


In [ ]:
# menjalankan one hot encoding pada category
category = pd.get_dummies(df.category)
category

,finance,food,health,hot,inet,news,oto,sport,travel
0,1,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
91012,0,0,0,0,0,0,0,0,1
91013,0,0,0,0,0,0,0,0,1
91014,0,0,0,0,0,0,0,0,1
91015,0,0,0,0,0,0,0,0,1


In [ ]:
# menggabungkan dataset dengan hasil one hot encoding
join_df = pd.concat([df, category], axis=1)
join_df = join_df.drop(columns='category')
join_df

,title,finance,food,health,hot,inet,news,oto,sport,travel
0,Kemnaker Awasi TKA di Meikarta,1,0,0,0,0,0,0,0,0
1,BNI Digitalkan BNI Java Jazz 2020,1,0,0,0,0,0,0,0,0
2,"Terbang ke Australia, Edhy Prabowo Mau Genjot ...",1,0,0,0,0,0,0,0,0
3,OJK Siapkan Stimulus Ekonomi Antisipasi Dampak...,1,0,0,0,0,0,0,0,0
4,Saran Buat Anies-RK yang Mangkir Rapat Banjir ...,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
91012,"Ketumpahan Air Panas di Pesawat, Kamu Bisa Tun...",0,0,0,0,0,0,0,0,1
91013,Foto: Bali & 9 Destinasi Paling Instagramable ...,0,0,0,0,0,0,0,0,1
91014,Game Bikin Turis Ini Liburan ke Jepang untuk.....,0,0,0,0,0,0,0,0,1
91015,"Sekeluarga Didepak dari Pesawat, Maskapai Bila...",0,0,0,0,0,0,0,0,1


In [ ]:
print(join_df['title'][91014])

Game Bikin Turis Ini Liburan ke Jepang untuk... Cari Yakuza!


In [ ]:
# merubah dataframe menjadui numpy array
judul = join_df['title'].values
label = join_df[['finance', 'food', 'health', 'hot', 'inet', 'news', 'oto', 'sport', 'travel']].values

In [ ]:
# membagi data training dan data testing
X_train, X_test,y_train,y_test = train_test_split(judul, label, test_size=0.2,random_state=42)

In [ ]:
# Inisialisasi Tokenizer dengan membatasi jumlah kata unik yang akan diberikan nomor indeks hingga 5000.
# Jika ada kata yang tidak termasuk dalam 5000 kata unik, mereka akan digantikan dengan token '-'
tokenizer = Tokenizer(num_words=5000, oov_token='-')

# Melatih Tokenizer dengan data training
tokenizer.fit_on_texts(X_train)

# Mengubah teks data training menjadi sekuens token (bilangan bulat)
sekuens_latih = tokenizer.texts_to_sequences(X_train)

# Mengubah teks data testing menjadi sekuens token (bilangan bulat)
sekuens_test = tokenizer.texts_to_sequences(X_test)

# Melakukan proses padding pada sekuens token data training
# Padding digunakan untuk membuat panjang setiap sekuens menjadi sama
# Dalam kasus ini, panjang sekuens adalah variabel dan mengikuti panjang kalimat terpanjang dalam data training
padded_latih = pad_sequences(sekuens_latih, padding='post', truncating='post')

# Melakukan proses padding pada sekuens token data testing
# Padding dilakukan agar sekuens token data testing memiliki panjang yang sama dengan sekuens data training
padded_test = pad_sequences(sekuens_test, padding='post', truncating='post')

In [ ]:
#membuat model dengan layer embedding dan menjalankan fungsi compile

model = tf.keras.Sequential([
    # Lapisan Embedding digunakan untuk mengubah bilangan bulat menjadi vektor yang padat dan terdistribusi
    # Input model akan memiliki 5000 kata unik (input_dim=5000) dengan dimensi keluaran 16 (output_dim=16)
    tf.keras.layers.Embedding(input_dim=5000, output_dim=16),

    # Lapisan LSTM (Long Short-Term Memory) adalah lapisan recurrent neural network
    # LSTM digunakan untuk memproses data sekuens dan menghasilkan output dengan dimensi 64
    tf.keras.layers.LSTM(64),

    # Lapisan Dense adalah lapisan fully connected dengan 128 unit dan fungsi aktivasi ReLU
    tf.keras.layers.Dense(128, activation='relu'),

    # Lapisan Dropout digunakan untuk menghindari overfitting dengan mematikan sejumlah unit acak selama pelatihan
    # Dropout rate adalah 0.2, yang berarti 20% dari unit akan dinonaktifkan selama pelatihan
    tf.keras.layers.Dropout(0.2),

    # Lapisan Dense lainnya dengan 64 unit dan fungsi aktivasi ReLU
    tf.keras.layers.Dense(64, activation='relu'),

    # Lapisan Dropout lagi dengan dropout rate 0.2
    tf.keras.layers.Dropout(0.2),

    # Lapisan Dense terakhir dengan 9 unit dan fungsi aktivasi softmax
    # Digunakan untuk output klasifikasi multikelas dengan 9 kelas
    # Output dari lapisan ini adalah probabilitas untuk masing-masing kelas
    tf.keras.layers.Dense(9, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# menerapkan kelas callbacks
class myCallback(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs={}):
    if(logs.get('accuracy')>0.95):
      print("/nAkurasi telah mencapai >95%!")
      self.model.stop_training = True
callbacks = myCallback()

In [ ]:
num_epochs = 30
history = model.fit(padded_latih, y_train, epochs=num_epochs,
                    validation_data=(padded_test, y_test), verbose=1, batch_size=16,
                    callbacks=[callbacks])

Epoch 1/30
4551/4551 [==============================] - 78s 17ms/step - loss: 1.1282 - accuracy: 0.6191 - val_loss: 0.7652 - val_accuracy: 0.7550
Epoch 2/30
4551/4551 [==============================] - 75s 17ms/step - loss: 0.6514 - accuracy: 0.7981 - val_loss: 0.5993 - val_accuracy: 0.8135
Epoch 3/30
4551/4551 [==============================] - 74s 16ms/step - loss: 0.5280 - accuracy: 0.8380 - val_loss: 0.5461 - val_accuracy: 0.8270
Epoch 4/30
4551/4551 [==============================] - 75s 17ms/step - loss: 0.4681 - accuracy: 0.8551 - val_loss: 0.5400 - val_accuracy: 0.8312
Epoch 5/30
4551/4551 [==============================] - 74s 16ms/step - loss: 0.4320 - accuracy: 0.8655 - val_loss: 0.5302 - val_accuracy: 0.8349
Epoch 6/30
4551/4551 [==============================] - 74s 16ms/step - loss: 0.4009 - accuracy: 0.8750 - val_loss: 0.5616 - val_accuracy: 0.8307
Epoch 7/30
4551/4551 [==============================] - 75s 16ms/step - loss: 0.3760 - accuracy: 0.8816 - val_loss: 0.5491 -

In [ ]:

from tensorflow.keras.preprocessing.sequence import pad_sequences

# Gunakan tokenizer yang telah Anda definisikan sebelumnya
tokenizer = Tokenizer(num_words=5000, oov_token='x')
tokenizer.fit_on_texts(judul)


In [ ]:
# Ganti contoh judul baru dengan judul yang ingin Anda uji
judul_baru = ['Saran Buat Anies-RK yang Mangkir Rapat Banjir di DPR', 'Game Bikin Turis Ini Liburan ke Jepang','Kemnaker Awasi TKA di Meikarta	']
sekuens_baru = tokenizer.texts_to_sequences(judul_baru)
padded_baru = pad_sequences(sekuens_baru)

import numpy as np

# Lakukan prediksi pada data inputan baru
predictions = model.predict(padded_baru)

# Mengambil indeks label dengan nilai probabilitas tertinggi
predicted_labels = np.argmax(predictions, axis=1)

# Ganti daftar nama label dengan label yang sesuai dengan model Anda
nama_label = ['finance', 'food', 'health', 'hot', 'inet', 'news', 'oto', 'sport', 'travel']

# Menampilkan hasil prediksi
for idx, judul in enumerate(judul_baru):
    print(f"Judul: {judul}")
    print(f"Predicted Category: {nama_label[predicted_labels[idx]]}")
    print("------------------------")


1/1 [==============================] - 1s 626ms/step
Judul: Saran Buat Anies-RK yang Mangkir Rapat Banjir di DPR
Predicted Category: news
------------------------
Judul: Game Bikin Turis Ini Liburan ke Jepang
Predicted Category: oto
------------------------
Judul: Kemnaker Awasi TKA di Meikarta	
Predicted Category: news
------------------------
